
# C7-cnn-transfer — Practice p19

**Type:** challenge · **Difficulty:** advanced · **Concepts:** transfer-learning, layer-freezing, nn-module, requires-grad, parameter-counting

The two-zone transfer build: a reusable freezing tool, surgery, and a
partial thaw — the construction used when the deepest features should
*also* be adjustable, not just the head.

**(a) The tool.**
Implement `freeze_except(module, prefixes)`:

- `prefixes` is a tuple of name prefixes;
- after the call, a parameter `p` with qualified name `n` (from
  `module.named_parameters()`) has
  `p.requires_grad == n.startswith(prefixes)` — thawed iff its name
  starts with any given prefix, frozen otherwise;
- returns `None`; one pass over `named_parameters()`.

**(b) The build.**
On `surg`, a **deepcopy** of `model`: graft
`surg.fc = nn.Linear(2048, 17)`, then call
`freeze_except(surg, ("layer4", "fc"))`.
(Note the order is safe *because* `freeze_except` sets both
directions explicitly — unlike a bare freeze loop, it cannot eat the
head.)

**(c) Hand predictions.**
Literal arithmetic only — plain int literals, or a chain of int
literals combined with `+`/`-` (as in `25_557_032 - 2_049_000 - ...`);
no code-derived count of ANY kind (**`numel`, `p.nelement()`,
`np.prod(p.shape)`, and every disguised helper — banned in this cell,
zero points**):

- `hand_trainable` — `layer4`'s scalars plus the fresh head's (the
  $2049k$ form at $k = 17$; the per-child table for `layer4`);
- `hand_frozen` — everything else (derive it from the model total
  *minus the old head* plus-and-minus the right pieces — show the
  chain).

**(d) Audit.**
`n_trainable`, `n_frozen` over `surg.parameters()` (**`numel`
allowed here**); `counts_match` against (c);
`trainable_tops` — sorted top-level prefixes owning any trainable
parameter (expect `['fc', 'layer4']`);
`out_shape` — seeded-batch output under `inference_mode` (expect
`(2, 17)`).

**(e) Why thaw `layer4` at all?** (markdown, 2–3 sentences)
Answer from the feature hierarchy: which depths carry the most
task-specific features, and why does that make the *deepest* stage
the natural candidate for adjustment when (beyond this course's
fence) training resources exist — while the head alone suffices for
the smallest projects?

**Banned (zero points): `numel`/`torchsummary`/`state_dict`-size
reads in cell (c); `torchvision.models.feature_extraction`; forward
hooks.**


In [ ]:
# Cache pin (course convention, plan 009): pretrained weights live in the repo's
# gitignored reference/cache/ -- resolve it from the repo root BEFORE importing torch.
import os, pathlib
_env_root = os.environ.get("USAAIO_BOOK_ROOT")
if _env_root:
    _root = pathlib.Path(_env_root).resolve()
else:
    _start = pathlib.Path.cwd().resolve()
    _root = next(
        p for p in [_start, *_start.parents]
        if (p / "syllabus.md").is_file() and (p / "curriculum").is_dir()
    )
os.environ["TORCH_HOME"] = str(_root / "reference" / "cache" / "torch")

import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

# float32 register (course exception): pretrained resnet50 is a float32 artifact.
# No float64 default here; inputs are cast .to(torch.float32) at the model
# boundary; float comparisons state atol=1e-6 / rtol=1e-5.
SEED = 20260804

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval()
assert next(model.parameters()).dtype == torch.float32

import copy

torch.manual_seed(SEED)
x = torch.randn(2, 3, 224, 224).to(torch.float32)


# (a) YOUR CODE HERE -- def freeze_except(module, prefixes): ...


# (b)
surg = ...  # YOUR CODE HERE


In [ ]:

# (c) hand arithmetic only -- numel banned in this cell
hand_trainable = ...  # YOUR CODE HERE (literal arithmetic; show the two terms)
hand_frozen = ...     # YOUR CODE HERE (literal arithmetic; show the chain)


In [ ]:

# (d) audit -- numel allowed here
n_trainable = ...     # YOUR CODE HERE
n_frozen = ...        # YOUR CODE HERE
counts_match = ...    # YOUR CODE HERE
trainable_tops = ...  # YOUR CODE HERE
out_shape = ...       # YOUR CODE HERE



*Your (e) answer here (2–3 sentences).*
